# 02 Clean the data and split it

Two fixes from 01 (the value 3 and the copies), then I split the wafers into three parts:

- **training (70%)**: the model learns from these
- **validation (15%)**: I use these to compare models and settings
- **test (15%)**: the final exam, used once at the end in 05

Saves `output/split.npz`.

In [1]:
import os # for making the output folder

import numpy as np
from sklearn.model_selection import train_test_split

import wafer

data = np.load("../data/Wafer_Map_Datasets.npz")
maps = data["arr_0"]
labels = data["arr_1"]
print(f"Wafers at the start: {len(maps):,}")

Wafers at the start: 38,015


## 1. Take out the wafers with a 3

The description doesn't say what 3 means, so I can't know if it's a good or a failed chip. There are only a few,
so I leave those wafers out.

In [2]:
has_3 = (maps == 3).any(axis=(1, 2)) # per wafer: is there a 3 anywhere in its rows and columns
maps = maps[~has_3] # ~ = not, so keep the ones without a 3
labels = labels[~has_3]
print(f"Removed {has_3.sum()} wafers, {len(maps):,} left")

Removed 105 wafers, 37,910 left


## 2. Take out the copies

`np.unique` gives the position of the first time every map shows up. I keep only those.

In [3]:
flat = maps.reshape(len(maps), -1)
# return_index gives where each different map shows up first.
# the _ is the stuff i dont need
_, first = np.unique(flat, axis=0, return_index=True)
first = np.sort(first) # unique mixes up the order, this puts it back

print(f"Removed {len(maps) - len(first):,} copies, {len(first):,} left")
maps = maps[first]
labels = labels[first]

Removed 1,020 copies, 36,890 left


## 3. Split 70 / 15 / 15

`train_test_split` shuffles the wafers and cuts them in two. I use it twice: first 70% off for training, then the
rest in half for validation and test.

`stratify` keeps the same mix of types in every part, so even the smallest type is in the test.
`random_state=42` makes the shuffle the same every time I run it.

In [4]:
# every wafer's type as a name like "Center + Scratch", so the split keeps the mix the same
types = [wafer.label_name(label) for label in labels]

# first cut: 70% training, 30% rest. stratify = same mix of types in both. 42 = same shuffle every run
maps_train, maps_rest, y_train, y_rest, types_train, types_rest = train_test_split(
    maps, labels, types, test_size=0.30, stratify=types, random_state=42)

# second cut: the rest in half, 15% validation 15% test
maps_val, maps_test, y_val, y_test = train_test_split(
    maps_rest, y_rest, test_size=0.50, stratify=types_rest, random_state=42)

print(f"Training: {len(maps_train):,}")
print(f"Validation: {len(maps_val):,}")
print(f"Test: {len(maps_test):,}")
print("Types in the test part:", len(np.unique(y_test, axis=0)))

Training: 25,823
Validation: 5,533
Test: 5,534
Types in the test part: 38


**What to look for:** about 70/15/15, and all 38 types in the test part.

## 4. Save

In [5]:
os.makedirs("output", exist_ok=True) # exist_ok so it doesnt crash if the folder is already there
np.savez_compressed("output/split.npz",
                    maps_train=maps_train, y_train=y_train,
                    maps_val=maps_val, y_val=y_val,
                    maps_test=maps_test, y_test=y_test) # name=array, read back later with ["name"]
print("Saved output/split.npz")

Saved output/split.npz
